# BigM study, using A.I

## Introduction .

I will use A.I to clarify how to use BigM in linear optimizations . I have noticed that ChatGpt has nice answers.



- Basic product mix
- Big-M: activate production
- Big-M: minimum batch
- Big-M: fixed cost
- Big-M: conditional constraint
- Big-M: either/or
- Big-M: conditional equality
- Summary of the patterns

Pan Li - Chinese university of Hong Kong (So far, the best doc I've found for now):

- https://www.math.cuhk.edu.hk/course_builder/1415/math3220/L2%20(without%20solution).pdf



# Summary of the patterns

In practice

For product-mix problems, the most useful Big-M applications are usually:

Product activation → don't produce unless product is selected<br>
Minimum batch size → if selected, produce at least X<br>
Fixed setup cost → pay setup cost when production starts<br>
Machine activation → machine capacity applies only if machine is opened<br>
Warehouse/factory opening → transportation only allowed if facility is open<br>
Logical rules → "if A, then B"<br>
Either/or decisions → choose one strategy<br>
Conditional constraints → constraint applies only under a certain decision<br><br>

If you're learning DOcplex/MILP, I'd recommend learning these in this order: activation → fixed cost → minimum batch → if-then → either/or. Those five cover a huge proportion of practical Big-M models.

# Activate/Deactivate a constraint - Production line activation

Absolutely. A Big-M constraint is easiest to understand when we add a yes/no decision to a normal linear product-mix problem.

Simple example: Product mix + Big-M

Imagine a small factory makes two products:

Product A → profit = €40/unit<br>
Product B → profit = €30/unit<br>

The factory has:

100 hours of machine time
80 units of raw material

|               | Product A | Product B | Available |
| ------------- | --------: | --------: | --------: |
| Machine hours |         2 |         1 |       100 |
| Raw material  |         1 |         2 |        80 |
| Profit        |       €40 |       €30 |           |


1. Normal linear product-mix model

Let:

\(x_A\) = quantity of Product A
\(x_B\) = quantity of Product B

Maximize profit:

$$ \max Z = 40x_A + 30x_B $$

Subject to:

$$ 2x_A+x_B \leq 100 $$ $$ x_A+2x_B \leq 80 $$ $$ x_A,x_B\geq0 $$

So far, this is just a Linear Programming (LP) problem.

2. Where does Big-M come in?

Suppose the factory has a rule:

If we produce Product B, we must open a special production line.

Opening the line costs €500.

Introduce a binary variable:

$$ y = \begin{cases} 1 & \text{if the special line is opened}\\ 0 & \text{otherwise} \end{cases} $$

Now we need to say:

If \(y=0\), Product B must be 0.
If \(y=1\), Product B can be produced.


We use Big-M:

$$ x_B \leq M y $$

For example, choose:

$$ M=40 $$

Why 40?

Because the raw-material constraint means Product B can never exceed 40 units:

$$ 2x_B\leq80 $$

so:

$$ x_B\leq40 $$

Therefore:

xB≤40y

3. See how Big-M works
If the line is closed
$$ y=0 $$

Then:

$$ x_B\leq40(0) $$ $$ x_B\leq0 $$

Therefore:

$$ x_B=0 $$

Product B cannot be produced.

If the line is open
$$ y=1 $$

Then:

$$ x_B\leq40(1) $$ $$ x_B\leq40 $$

Product B can be produced.

That's the basic idea of Big-M:

Binary variable × large enough number = activate/deactivate a constraint.


4. Complete mathematical model

We now have:

Decision variables
$$ x_A,x_B\geq0 $$ $$ y\in\{0,1\} $$
Objective

Opening the special line costs €500:

$$ \max Z = 40x_A+30x_B-500y $$
Constraints

Machine:

$$ 2x_A+x_B\leq100 $$

Raw material:

$$ x_A+2x_B\leq80 $$

Big-M:

$$ x_B\leq40y $$

And:

$$ x_A,x_B\geq0 $$
y∈{0,1}

5. DOcplex Python code

Here is a very simple implementation:

In [2]:
from docplex.mp.model import Model

# Create model
mdl = Model("product_mix_bigM")

# -----------------------
# Decision variables
# -----------------------

xA = mdl.continuous_var(lb=0, name="Product_A")
xB = mdl.continuous_var(lb=0, name="Product_B")

# Binary variable:
# 1 = open special line
# 0 = do not open
y = mdl.binary_var(name="Open_Line")

# -----------------------
# Objective
# -----------------------

mdl.maximize(
    40 * xA +
    30 * xB -
    500 * y
)

# -----------------------
# Constraints
# -----------------------

# Machine hours
mdl.add_constraint(
    2 * xA + xB <= 100,
    "Machine"
)

# Raw material
mdl.add_constraint(
    xA + 2 * xB <= 80,
    "Raw_Material"
)

# Big-M constraint
M = 40

mdl.add_constraint(
    xB <= M * y,
    "BigM"
)

# -----------------------
# Solve
# -----------------------

solution = mdl.solve()

# -----------------------
# Display results
# -----------------------

if solution:
    print("Product A =", xA.solution_value)
    print("Product B =", xB.solution_value)
    print("Open Line =", y.solution_value)
    print("Profit =", mdl.objective_value)
else:
    print("No solution")

ModuleNotFoundError: No module named 'docplex'